In [27]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel
from pydantic import Field
from typing import Annotated
load_dotenv()
import operator

In [28]:
model=ChatGoogleGenerativeAI(model="gemini-3.6-flash")

In [29]:
class Eval(BaseModel):
    feedback:str=Field(description="detailed Feedback on the essay")
    score:int=Field(description="score out of 10",ge=0,le=10)
    

In [30]:
structured_model=model.with_structured_output(Eval)

In [31]:
essay="""Last September, India put a spacecraft into orbit around Mars. Since then, boasts about the country’s scientific prowess have grown outlandishly. In October, Prime Minister Narendra Modi pointed to the half-human, half-elephant Hindu god Ganesh as evidence that ancient Indians had pioneered the art of plastic surgery. Over the weekend, science and technology minister Harsh Vadhan told delegates to the Indian Science Congress—an annual gathering of the country’s top researchers—that Indian mathematicians had discovered the Pythagorean theorem and graciously allowed the Greeks to take credit. Other speakers claimed that bacteria in cow dung could turn objects into solid gold, and that 7,000 years ago, Indians were flying huge airplanes “from one planet to another.” As ludicrous as these claims are, what should really worry Indians is the current state of the country’s research sector. Despite high-profile successes such as the Mars mission, and its well-known prowess in information technology, India lags badly in technological research and development. Forget 7,000-year-old planes: After more than 30 years of trying, the country still hasn’t been able to develop an indigenous fighter aircraft—technology for which is widely available globally. India spends less than 1% of its gross domestic product (GDP) on R&D. China spends 2%, the US 2.8%, Japan 3.4% and Korea 4%. India’s share of global R&D stands at a dismal 2.7%—compared with 30% for the US. Even China now accounts for almost 15% of such spending, having doubled in total between 2008 and 2012. It isn’t entirely surprising that India lays out so little on research. A majority of R&D spending takes place in the manufacturing sector—particularly at the upper end of the value chain—and India has a very weak manufacturing base compared with the US, China, Japan and Korea. In principle, any policy changes that boosted manufacturing—Modi has promised to improve India’s ranking in the World Bank’s Ease of Doing Business Survey from 141 to 50—would also lead to increased R&D spending. At the same time, though, India gets less than it should out of the money that it does spend. The fighter-jet project has stumbled along in part because state-controlled Hindustan Aeronautics Ltd has a monopoly on the plane’s manufacture. The company suffers from all the red tape and inefficiency that plagues the rest of India’s huge public sector. It lacks the autonomy to make bold decisions. It can’t attract top engineering talent with its rigid (and low) salary scales. At the Science Congress, Modi emphasized the need to trim back this kind of red tape. The government should also loosen its stranglehold over state-run laboratories and technical universities such as the famed Indian Institutes of Technology (IIT). Only last week, the head of Delhi’s IIT resigned under pressure from the ministry of education. That kind of micro-management drives talent away and shifts the focus from science and technology—the IITs’ core competence—to managing politics. Of course, the government has an important role to play in promoting basic research. But ideally the state should act more as a facilitator, encouraging greater cooperation between academia, laboratories and private industry and where necessary supporting R&D through financial resources (including for higher salaries to attract talent), but without managerial interference. In the US, federal government spending on R&D peaked at around 1.2% of GDP in the late 1980s. While it’s since dropped under 1%, even now 63% of the funding for academic R&D in the US comes from the government. In other spheres like defence, the government supports research by being a big buyer of high-tech equipment. For India, foreign investment should provide another key source of funding. The country boasts a strong base of trained scientists and engineers available for a fraction of the cost in advanced economies. To fulfill that potential, however, the country needs to continue strengthening its weak patents regime and improving what remains a generally hostile investment for foreign businesses. There’s little time to waste—and not just because pseudo-scientific quackery seem to be on the rise. India set up its first IIT in 1950, at a time when Korea was still wracked by civil war and China was about to embark on decades of Maoist chaos. If the country is to catch up to its Asian peers, it needs to start now

"""

In [ ]:
#prompt=f'Evaluate the language quality of the following essay and provide a detailed feedback and a score out of 10. The essay is: {essay}'

In [ ]:
#structured_model.invoke(prompt).score

9

In [ ]:
#structured_model.invoke(prompt).feedback

"The essay demonstrates exceptionally high language quality, characterized by advanced vocabulary, strong coherence, and a compelling argumentative tone typical of top-tier journalistic writing. Complex economic and policy concepts are articulated clearly, backed by appropriate statistics and logical transitions. Grammar, punctuation, and syntax are virtually flawless throughout, effectively contrasting pseudoscientific rhetoric with pragmatic analysis of India's R&D challenges."

In [35]:
class UPSC(TypedDict):
    essay:str
    language_feedback:str
    analysis_feedback:str
    clarity_feedback:str
    overall_feedback:str
    individual_scores:Annotated[list[int], operator.add]
    avg_score:float

In [36]:
def evaluate_language(state: UPSC):
    prompt=f'Evaluate the language quality of the following essay and provide a detailed feedback and a score out of 10. The essay is: {state["essay"]}'
    result=structured_model.invoke(prompt)
    return {"language_feedback":result.feedback,"individual_scores":[result.score]}

In [37]:
def evaluate_analysis(state: UPSC):
    prompt=f'Evaluate the depth of analysis of the following essay and provide a detailed feedback and a score out of 10. The essay is: {state["essay"]}'
    result=structured_model.invoke(prompt)
    return {"analysis_feedback":result.feedback,"individual_scores":[result.score]}

In [38]:
def evaluate_thought(state: UPSC):
    prompt=f'Evaluate the clarity of thought of the following essay and provide a detailed feedback and a score out of 10. The essay is: {state["essay"]}'
    result=structured_model.invoke(prompt)
    return {"clarity_feedback":result.feedback,"individual_scores":[result.score]}

In [51]:
def final_evaluation(state:UPSC):
    ## summary feedback
    prompt = f"""
    Provide a summary feedback for the following essay based on the evaluations:
    {state["language_feedback"]},
    {state["analysis_feedback"]},
    {state["clarity_feedback"]}
    """
    overall_feedback = model.invoke(prompt)
    avg_score = sum(state["individual_scores"]) / len(state["individual_scores"])
    return {
        "overall_feedback": overall_feedback.content,
        "avg_score": avg_score
    }

In [52]:
graph=StateGraph(UPSC)

graph.add_node('evaluate_language',evaluate_language)
graph.add_node('evaluate_analysis',evaluate_analysis)
graph.add_node('evaluate_thought',evaluate_thought)
graph.add_node('final_evaluation',final_evaluation)

graph.add_edge(START,'evaluate_language')
graph.add_edge(START,'evaluate_analysis')
graph.add_edge(START,'evaluate_thought')
graph.add_edge('evaluate_language','final_evaluation')
graph.add_edge('evaluate_analysis','final_evaluation')
graph.add_edge('evaluate_thought','final_evaluation')
graph.add_edge('final_evaluation',END)

In [53]:
workflow=graph.compile()


In [54]:
initial_state={
    'essay':essay
}
result=workflow.invoke(initial_state)

ChatGoogleGenerativeAIError: Error calling model 'gemini-3.6-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 20.247134704s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

In [ ]:
print(result)